# gf_mul LRA Leakage Assessment

저장된 trace HDF5 파일을 불러와 Operand 0 / Operand 1 / Output 각각에 대해 **bit-level linear regression (R²)** 을 계산한다.

논문 Figure 3에 대응.


## 1. Imports

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## 2. Configuration

In [ ]:
# === user config ===
H5_PATH = "/Users/ijaeyeon/chipwhisperer/jupyter/user/batches_gf_mul_CW308_STM32F4/100000trace_1400point_seed12.h5"

# Leave these as None to auto-detect from the HDF5 file.
TRACE_KEY = None
A_KEY = None
B_KEY = None
V_KEY = None

USE_SUBSET = False
MAX_TRACES = 50000

FIGSIZE = (12, 4)
SMOOTH = False
SMOOTH_WIN = 11

## 3. HDF5 key resolver

In [ ]:
def resolve_h5_keys(f, trace_key=None, a_key=None, b_key=None, v_key=None):
    keys = list(f.keys())

    def pick(user_key, candidates, name):
        if user_key is not None:
            if user_key not in keys:
                raise KeyError(
                    f"{name} key '{user_key}' not found. Available datasets: {keys}"
                )
            return user_key
        for cand in candidates:
            if cand in keys:
                return cand
        raise KeyError(
            f"Could not auto-detect {name} key. Available datasets: {keys}"
        )

    trace_key = pick(trace_key, ["traces", "trace", "waves", "wave"], "trace")
    a_key     = pick(a_key,     ["a", "op0", "operand0"], "a")
    b_key     = pick(b_key,     ["b", "op1", "operand1"], "b")
    v_key     = pick(v_key,     ["v", "out", "output"],   "v")
    return trace_key, a_key, b_key, v_key

## 4. Load traces

In [ ]:
assert Path(H5_PATH).exists(), f"File not found: {H5_PATH}"

with h5py.File(H5_PATH, "r") as f:
    print("datasets:", list(f.keys()))
    TRACE_KEY, A_KEY, B_KEY, V_KEY = resolve_h5_keys(
        f, TRACE_KEY, A_KEY, B_KEY, V_KEY
    )
    print("resolved keys:",
          {"TRACE_KEY": TRACE_KEY, "A_KEY": A_KEY, "B_KEY": B_KEY, "V_KEY": V_KEY})

    n_total = f[TRACE_KEY].shape[0]
    n_samples = f[TRACE_KEY].shape[1]
    print(f"N traces  = {n_total}")
    print(f"N samples = {n_samples}")

    if USE_SUBSET:
        n_use = min(MAX_TRACES, n_total)
    else:
        n_use = n_total

    traces = f[TRACE_KEY][:n_use].astype(np.float64)
    a = f[A_KEY][:n_use].astype(np.uint8)
    b = f[B_KEY][:n_use].astype(np.uint8)
    v = f[V_KEY][:n_use].astype(np.uint8)

print("Loaded:")
    
print(" traces:", traces.shape, traces.dtype)
print(" a/b/v :", a.shape, b.shape, v.shape)

## 5. Bit-decomposition & smoothing helpers

In [ ]:
def bits8(x: np.ndarray) -> np.ndarray:
    """
    x: uint8 vector of shape (N,)
    return: bit matrix of shape (N, 8), LSB first
    """
    x = x.astype(np.uint8)
    return ((x[:, None] >> np.arange(8)) & 1).astype(np.float64)

def hw8(x: np.ndarray) -> np.ndarray:
    """Hamming weight of uint8 vector."""
    return np.unpackbits(x[:, None], axis=1).sum(axis=1).astype(np.float64)

def moving_average_1d(y: np.ndarray, w: int) -> np.ndarray:
    if w <= 1:
        return y.copy()
    k = np.ones(w, dtype=np.float64) / w
    return np.convolve(y, k, mode="same")

## 6. LRA (bit model → R² per sample)

In [ ]:
def lra_r2_bitmodel(traces: np.ndarray, labels_u8: np.ndarray) -> np.ndarray:
    """
    traces: shape (N, S)
    labels_u8: shape (N,) uint8
    returns: R^2 curve of shape (S,)

    Model:
        x_i = beta0 + sum_{j=1}^8 beta_j * y_{i,j} + eps
    computed independently for each sample point.
    """
    X = traces.astype(np.float64)              # (N, S)
    Y = bits8(labels_u8)                       # (N, 8)

    # intercept is handled by centering
    Xc = X - X.mean(axis=0, keepdims=True)     # (N, S)
    Yc = Y - Y.mean(axis=0, keepdims=True)     # (N, 8)

    # Solve B = (Y^T Y)^(-1) Y^T X
    G = Yc.T @ Yc                              # (8, 8)
    Ginv = np.linalg.pinv(G)                   # robust inverse
    B = Ginv @ (Yc.T @ Xc)                     # (8, S)

    Xhat = Yc @ B                              # (N, S)

    sse = np.sum((Xc - Xhat) ** 2, axis=0)     # (S,)
    sst = np.sum(Xc ** 2, axis=0)              # (S,)

    r2 = np.zeros_like(sse)
    nz = sst > 0
    r2[nz] = 1.0 - (sse[nz] / sst[nz])
    r2[~nz] = 0.0
    return r2

## 7. Compute R² for operand 0 / operand 1 / output

In [ ]:
r2_a = lra_r2_bitmodel(traces, a)
r2_b = lra_r2_bitmodel(traces, b)
r2_v = lra_r2_bitmodel(traces, v)

if SMOOTH:
    r2_a_plot = moving_average_1d(r2_a, SMOOTH_WIN)
    r2_b_plot = moving_average_1d(r2_b, SMOOTH_WIN)
    r2_v_plot = moving_average_1d(r2_v, SMOOTH_WIN)
else:
    r2_a_plot = r2_a
    r2_b_plot = r2_b
    r2_v_plot = r2_v

print("done")
print("max R2 operand 0 =", float(np.max(r2_a)))
print("max R2 operand 1 =", float(np.max(r2_b)))
print("max R2 output    =", float(np.max(r2_v)))

## 8. Plot

In [ ]:
plt.figure(figsize=FIGSIZE)
plt.plot(r2_a_plot, label="Operand 0")
plt.plot(r2_b_plot, label="Operand 1")
plt.plot(r2_v_plot, label="Output")
plt.xlabel("Sample")
plt.ylabel(r"$R^2$")
plt.title("LRA leakage assessment on gf_mul")
plt.legend()
plt.grid(alpha=0.2)
plt.show()